# TinySTT: train the acoustic model on a Kaggle GPU

Trains TinySTT's streaming CTC phoneme model on LibriSpeech train-clean-100 and train-clean-360
(pronunciations from the LibriSpeech lexicon, so no utterance is skipped), builds a 40k-word
lexicon and a bigram LM from the transcripts, and packs everything into one model container
`tinystt.tstt` (in `/kaggle/working`) for the C++ library. See `docs/training.md`.

**Settings:** Accelerator *GPU T4* (or P100), Internet *On*. Run with *Save Version -> Save & Run
All (Commit)*: it runs in the background (Kaggle limit 12 h); `MAX_HOURS` stops training early
enough for QAT, export and evaluation.

**Data:** attach the Kaggle dataset `victorling/librispeech-clean` (LibriSpeech, extracted): no
download. Without it the subsets are downloaded from openslr.org.

**CPU first:** run once without GPU: that validates the notebook on a subset and keeps
LibriSpeech in the output. Then add this notebook's output as input, switch the GPU on and run
again: the full training reuses the data.

**Code:** attach the dataset `tinystt-research` (TinySTT's `research/` folder), or the notebook
clones the GitHub repository.

## 1. Settings and code

In [ ]:
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/pschatzmann/TinySTT"
WORK = "/kaggle/working"
MAX_HOURS = 10.5                  # training time budget (QAT + export follow)
EPOCHS = 7                        # 460 h of audio: about 80 min per epoch on a T4
QAT_EPOCHS = 1
LEXICON_WORDS = 40000
TRAIN_SUBSETS = ["train-clean-100", "train-clean-360"]
# pronunciations for the training targets: the LibriSpeech lexicon covers every
# LibriSpeech word (CMUdict + letter-to-sound), so no utterance is skipped
PRON_LEXICON = "librispeech-lexicon.txt"
PRON_URL = "https://openslr.elda.org/resources/11/librispeech-lexicon.txt"

import glob
import torch
GPU = torch.cuda.is_available()
# Without a GPU this is a smoke run: it validates the whole notebook on a
# small subset and keeps LibriSpeech in the output, so that a GPU run can
# attach this notebook's output as input and skip the download.
SMOKE = not GPU
# attached LibriSpeech subsets (e.g. the dataset victorling/librispeech-clean or
# a previous run's output) are linked into a writable folder; only missing
# subsets are downloaded
found = {}
for d in sorted(glob.glob("/kaggle/input/**/LibriSpeech/*-*", recursive=True)):
    if os.path.isdir(d):
        found.setdefault(os.path.basename(d), d)
attached = sorted(found)
# a smoke run without attached data downloads into the output (saved for the
# GPU run, at most 20 GB: train-clean-100 only); otherwise local disk
DATA = f"{WORK}/librispeech" if SMOKE and not attached else "/tmp/librispeech"
if DATA.startswith(WORK):
    TRAIN_SUBSETS = ["train-clean-100"]
os.makedirs(f"{DATA}/LibriSpeech", exist_ok=True)
for name, d in found.items():
    dst = f"{DATA}/LibriSpeech/{name}"
    if not os.path.exists(dst):
        os.symlink(d, dst)
print(f"GPU: {GPU}, smoke run: {SMOKE}, LibriSpeech: {DATA}, attached: {attached}, training on {TRAIN_SUBSETS}")
TRAIN_ARGS = (["--max-items", "300", "--epochs", "2", "--qat-epochs", "1", "--max-hours", "0.5"] if SMOKE
              else ["--epochs", str(EPOCHS), "--qat-epochs", str(QAT_EPOCHS), "--max-hours", str(MAX_HOURS)])
EVAL_ITEMS = 50 if SMOKE else 500
SUBSETS = " ".join(TRAIN_SUBSETS)

os.makedirs(WORK, exist_ok=True)
# find the attached code dataset wherever Kaggle mounts it
# (/kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>/...)
import glob
hits = sorted(glob.glob("/kaggle/input/**/prepare_librispeech.py", recursive=True))
print("mounted inputs:", hits)
if hits:
    shutil.copytree(os.path.dirname(hits[0]), f"{WORK}/research", dirs_exist_ok=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/tmp/TinySTT"])
    if r.returncode != 0:
        raise SystemExit("TinySTT code not found: attach the 'tinystt-research' dataset (Add Input)")
    shutil.copytree("/tmp/TinySTT/research", f"{WORK}/research", dirs_exist_ok=True)
R = f"{WORK}/research"
# the dataset ships cmudict.dict at its root; load_cmudict() looks in research/data
os.makedirs(f"{R}/data", exist_ok=True)
if os.path.exists(f"{R}/cmudict.dict"):
    shutil.move(f"{R}/cmudict.dict", f"{R}/data/cmudict.dict")
LEX = f"{R}/{PRON_LEXICON}"
if not os.path.exists(LEX):
    subprocess.run(["curl", "-sS", "-o", LEX, PRON_URL], check=True)
print(sorted(os.listdir(R)))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile"], check=True)
if GPU:
    subprocess.run(["nvidia-smi"])

## 2. LibriSpeech (download, extract, manifests)

In [ ]:
!python -u {R}/prepare_librispeech.py --root {DATA} --out {WORK}/manifests --subsets {SUBSETS} dev-clean test-clean --delete-archive
!df -h /tmp {WORK}

## 3. Train (CTC, last epochs quantization-aware) and export

In [ ]:
train_args = " ".join(TRAIN_ARGS)
train_manifests = " ".join(f"{WORK}/manifests/{s}.tsv" for s in TRAIN_SUBSETS)
!python -u {R}/train.py --train {train_manifests} --dev {WORK}/manifests/dev-clean.tsv --lexicon {LEX} \
    --out {WORK}/run --workers 4 {train_args} 2>&1 | tee {WORK}/train.log

## 4. Lexicon, language model, one container

In [ ]:
!python -u {R}/build_lexicon.py --vocab-from {WORK}/manifests/train-clean-100.tsv --max-words {LEXICON_WORDS} --out {WORK}/lexicon.tstt
!python -u {R}/build_lm.py --text-from {WORK}/manifests/train-clean-100.tsv --lexicon {WORK}/lexicon.tstt --out {WORK}/lm.tstt
!python -u {R}/pack.py {WORK}/run/model.tstt {WORK}/lexicon.tstt {WORK}/lm.tstt --out {WORK}/tinystt.tstt

## 5. Phoneme error rate on test-clean (greedy, int8 weights)

In [ ]:
!python -u {R}/eval.py --container {WORK}/tinystt.tstt --manifest {WORK}/manifests/test-clean.tsv --max-items {EVAL_ITEMS} | tee {WORK}/eval.txt
!ls -la {WORK}